In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
CLEANED_DATA = PROJECT_ROOT / "cleaned_data"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"
FEATURE_DATA = PROJECT_ROOT / "feature_data"

#results = pd.read_csv(CLEANED_DATA / "results_2000_onwards.csv")
team_matches = pd.read_csv(PROCESSED_DATA / "team_matches.csv")
match_feat = pd.read_csv(FEATURE_DATA / "match_feature_V1.csv")

In [2]:
team_matches["date"] = pd.to_datetime(team_matches["date"])
match_feat["date"] = pd.to_datetime(match_feat["date"])

print(team_matches["date"].dtype)
print(match_feat["date"].dtype)

datetime64[us]
datetime64[us]


In [3]:
team_matches = team_matches.sort_values("date").reset_index(drop = True)

In [4]:
def get_last5_stats(team, match_date):
    history = team_matches[
        (team_matches["team"] == team) &
        (team_matches["date"] < match_date)
    ]
    last5 = history.tail(5)

    if last5.empty:
        return 0.0, 0.0, 0.0
    
    points = (
        (last5["result"] == "W").sum() * 3 +
         (last5["result"] == "D").sum()
    )

    ppg = points / len(last5)
    
    avg_gf = last5["goals_for"].mean()
    avg_ga = last5["goals_against"].mean()

    return ppg, avg_gf, avg_ga

In [5]:
match_feat.iloc[1000]

date                   2000-11-19 00:00:00
home_team                         Djibouti
away_team                          Somalia
home_score                               0
away_score                               0
goal_difference                          0
home_elo                       1426.600265
away_elo                         1466.2359
elo_diff                        -39.635635
home_matches_played                      4
away_matches_played                      2
home_win_pct                           0.0
away_win_pct                           0.0
home_avg_gf                           0.75
away_avg_gf                            0.0
home_avg_ga                            3.5
away_avg_ga                            3.0
neutral                                  1
tournament                      CECAFA Cup
target                                   1
Name: 1000, dtype: object

In [6]:
row = match_feat.iloc[1000]

get_last5_stats(
    row["home_team"],
    row["date"]
)

(np.float64(0.25), np.float64(0.75), np.float64(3.5))

In [7]:
history = team_matches[
    (team_matches["team"] == row["home_team"]) &
    (team_matches["date"] < row["date"])
].tail(5)

history

,date,team,opponent,goals_for,goals_against,tournament,neutral,result
579,2000-04-07,Djibouti,Congo DR,1,1,FIFA World Cup qualification,False,D
718,2000-04-23,Djibouti,Congo DR,1,9,FIFA World Cup qualification,False,L
1236,2000-06-30,Djibouti,Burundi,1,3,African Cup of Nations qualification,False,L
1332,2000-07-15,Djibouti,Burundi,0,1,African Cup of Nations qualification,False,L


In [8]:
home_last5_ppg = []
away_last5_ppg = []

home_last5_avg_gf = []
away_last5_avg_gf = []

home_last5_avg_ga = []
away_last5_avg_ga = []

In [9]:
row = match_feat.iloc[1500]

home_ppg, home_gf, home_ga = get_last5_stats(
    row["home_team"],
    row["date"]
)

away_ppg, away_gf, away_ga = get_last5_stats(
    row["away_team"],
    row["date"]
)

print("Home:", row["home_team"], home_ppg, home_gf, home_ga)
print("Away:", row["away_team"], away_ppg, away_gf, away_ga)

Home: Korea Republic 2.0 1.4 1.4
Away: Mexico 1.4 1.2 1.4


In [10]:
history = team_matches[
    (team_matches["team"] == row["home_team"]) &
    (team_matches["date"] < row["date"])
].tail(5)

history

,date,team,opponent,goals_for,goals_against,tournament,neutral,result
2242,2001-02-11,Korea Republic,United Arab Emirates,4,1,United Arab Emirates Friendship Tournament,False,W
2747,2001-04-24,Korea Republic,IR Iran,1,0,Friendly,True,W
2818,2001-04-26,Korea Republic,Egypt,2,1,Friendly,False,W
2961,2001-05-25,Korea Republic,Cameroon,0,0,Friendly,False,D
2986,2001-05-30,Korea Republic,France,0,5,Confederations Cup,False,L


In [11]:
for _, row in match_feat.iterrows():

    home_ppg, home_gf, home_ga = get_last5_stats(
    row["home_team"],
    row["date"]
)
    away_ppg, away_gf, away_ga = get_last5_stats(
    row["away_team"],
    row["date"]
)
    home_last5_ppg.append(home_ppg)
    away_last5_ppg.append(away_ppg)
    
    home_last5_avg_gf.append(home_gf)
    away_last5_avg_gf.append(away_gf)
    
    home_last5_avg_ga.append(home_ga)
    away_last5_avg_ga.append(away_ga)

In [12]:
print(len(match_feat))
print(len(home_last5_ppg))
print(len(away_last5_ppg))

23760
23760
23760


In [13]:
match_feat["home_last5_ppg"] = home_last5_ppg
match_feat["away_last5_ppg"] = away_last5_ppg

match_feat["home_last5_avg_gf"] = home_last5_avg_gf
match_feat["away_last5_avg_gf"] = away_last5_avg_gf

match_feat["home_last5_avg_ga"] = home_last5_avg_ga
match_feat["away_last5_avg_ga"] = away_last5_avg_ga

In [14]:
match_feat[
    [
        "home_last5_ppg",
        "away_last5_ppg",
        "home_last5_avg_gf",
        "away_last5_avg_gf",
        "home_last5_avg_ga",
        "away_last5_avg_ga",
    ]
].isna().sum()

home_last5_ppg       0
away_last5_ppg       0
home_last5_avg_gf    0
away_last5_avg_gf    0
home_last5_avg_ga    0
away_last5_avg_ga    0
dtype: int64

In [15]:
match_feat.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'goal_difference', 'home_elo', 'away_elo', 'elo_diff',
       'home_matches_played', 'away_matches_played', 'home_win_pct',
       'away_win_pct', 'home_avg_gf', 'away_avg_gf', 'home_avg_ga',
       'away_avg_ga', 'neutral', 'tournament', 'target', 'home_last5_ppg',
       'away_last5_ppg', 'home_last5_avg_gf', 'away_last5_avg_gf',
       'home_last5_avg_ga', 'away_last5_avg_ga'],
      dtype='str')

In [16]:
match_feat.tail()

,date,home_team,away_team,home_score,away_score,goal_difference,home_elo,away_elo,elo_diff,home_matches_played,...,away_avg_ga,neutral,tournament,target,home_last5_ppg,away_last5_ppg,home_last5_avg_gf,away_last5_avg_gf,home_last5_avg_ga,away_last5_avg_ga
23755,2026-07-05,Mexico,England,2,3,-1,1914.033707,1964.601665,-50.567958,473,...,0.775701,0,FIFA World Cup,0,3.0,2.6,2.6,1.8,0.2,0.6
23756,2026-07-06,Portugal,Spain,0,1,-1,1940.716327,2011.585230,-70.868903,336,...,0.707246,1,FIFA World Cup,0,2.2,2.2,2.0,1.8,0.6,0.2
23757,2026-07-06,USA,Belgium,1,4,-3,1812.318122,1861.830427,-49.512304,423,...,1.044828,0,FIFA World Cup,0,1.8,2.2,2.2,2.8,1.2,0.8
23758,2026-07-07,Argentina,Egypt,3,2,1,2064.934264,1801.439090,263.495173,342,...,0.837535,1,FIFA World Cup,2,3.0,1.2,2.6,1.4,0.6,1.2
23759,2026-07-07,Switzerland,Colombia,0,0,0,1882.616182,1957.738735,-75.122554,290,...,0.868098,1,FIFA World Cup,1,2.2,2.6,2.0,1.4,0.8,0.2


In [17]:
match_feat.to_csv(FEATURE_DATA / "match_feature_V2.csv", index = False)

## Match Features V2 - Complete!!!

I added new features such as home/away last 5 points per game, home/away last 5 avg goals_for and home/away last 5 avg goals_against.

### Added Features
- `home_last5_ppg`
- `away_last5_ppg`
- `home_last5_avg_gf`
- `away_last5_avg_gf`
- `home_last5_avg_ga`
- `away_last5_avg_ga`

### Feature Logic
- Uses only matches played **before** the current match (`date < match_date`).
- Calculates statistics from each team's **last 5 matches** (or fewer if less history exists).
- No data leakage. (i checked :))

### Validation
- Manually verified multiple matches (Italy, Syria, etc.).
- All calculations (PPG, Avg GF, Avg GA) matched expected values.
- No missing values in the six new columns.
- Column lengths matched the dataset (`25,393` rows). (row count may vary because of new matches being still added.)

### Output
- Saved as `match_features_V2.csv`.

### Next in line
- Review final feature set.
- Perform chronological train/validation/test split.
- Begin baseline machine learning model.